In [3]:
from google.colab import files
uploaded = files.upload()

Saving text.csv.zip to text.csv.zip


In [4]:
import os
import zipfile
import pandas as pd

with zipfile.ZipFile('text.csv.zip', 'r') as zip_ref:
    zip_ref.extractall('emotion_data')

print(os.listdir('emotion_data'))

['text.csv']


In [5]:
df = pd.read_csv('emotion_data/text.csv')
print(df.shape)
print(df.head())
print(df.columns)

(416809, 3)
   Unnamed: 0                                               text  label
0           0      i just feel really helpless and heavy hearted      4
1           1  ive enjoyed being able to slouch about relax a...      0
2           2  i gave up my internship with the dmrg and am f...      4
3           3                         i dont know i feel so lost      0
4           4  i am a kindergarten teacher and i am thoroughl...      4
Index(['Unnamed: 0', 'text', 'label'], dtype='object')


In [6]:
print(df['label'].value_counts())

label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64


In [7]:
import re

# Drop unnecessary column
df = df.drop('Unnamed: 0', axis=1)

# Map numeric labels to emotion names
emotion_map = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
df['emotion'] = df['label'].map(emotion_map)

# Clean text: lowercase, remove special characters/numbers
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

print(df[['text', 'clean_text', 'emotion']].head())
print(df['emotion'].value_counts())

                                                text  \
0      i just feel really helpless and heavy hearted   
1  ive enjoyed being able to slouch about relax a...   
2  i gave up my internship with the dmrg and am f...   
3                         i dont know i feel so lost   
4  i am a kindergarten teacher and i am thoroughl...   

                                          clean_text  emotion  
0      i just feel really helpless and heavy hearted     fear  
1  ive enjoyed being able to slouch about relax a...  sadness  
2  i gave up my internship with the dmrg and am f...     fear  
3                         i dont know i feel so lost  sadness  
4  i am a kindergarten teacher and i am thoroughl...     fear  
emotion
joy         141067
sadness     121187
anger        57317
fear         47712
love         34554
surprise     14972
Name: count, dtype: int64


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['clean_text'])
y = df['emotion']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (333447, 5000)
Test shape: (83362, 5000)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8968594803387635

Classification Report:
               precision    recall  f1-score   support

       anger       0.90      0.90      0.90     11463
        fear       0.85      0.85      0.85      9542
         joy       0.91      0.93      0.92     28214
        love       0.81      0.77      0.79      6911
     sadness       0.94      0.93      0.93     24238
    surprise       0.78      0.71      0.75      2994

    accuracy                           0.90     83362
   macro avg       0.86      0.85      0.86     83362
weighted avg       0.90      0.90      0.90     83362


Confusion Matrix:
 [[10348   349   220    40   497     9]
 [  342  8127   201    31   443   398]
 [  154   144 26326  1074   378   138]
 [   54    22  1444  5303    76    12]
 [  602   398   590    77 22522    49]
 [   19   522   228    15    72  2138]]
